# OC2 Joint Oracle Full-Model Pilot

Notebook này là thí nghiệm cuối cùng để kiểm tra câu hỏi còn mở:

- `L4` có còn plastic đủ để sửa Failure B nếu dùng `joint objective` đúng không?
- Việc sửa B có giữ được lời giải cho Failure A dưới hard gate midband hay không?

Thiết kế:

- giữ `L4` làm main objective
- downweight mượt vùng `raw-center ambiguous` trong main loss
- xây `trusted oracle center bank` theo stratification 2D:
  - `raw |y|`
  - `|pred_L4|`
- fine-tune ngắn từ `L4` trên **toàn model**, không chỉ head
- dùng **một optimizer step duy nhất** cho `main + aux`
- chọn checkpoint bằng:
  - hard gate cho midband
  - rồi tối ưu center-only score

In [1]:
from dataclasses import asdict, replace
from pathlib import Path
import sys

import pandas as pd
import torch
from IPython.display import display

PROJECT_ROOT = Path(r"C:\Users\USER\Desktop\chess_engine")
EXPERIMENT_DIR = PROJECT_ROOT / "experiments" / "oc2_joint_oracle_full_model_pilot"
DATA_ROOT = PROJECT_ROOT / "data" / "process"
RUN_DIR = Path(r"C:\Users\USER\Downloads\dgrn_5m_v3_stage2_polish_run1")

if str(EXPERIMENT_DIR) not in sys.path:
    sys.path.insert(0, str(EXPERIMENT_DIR))

import oc2_joint_oracle_full_model_helpers as lab

torch.set_float32_matmul_precision("high")
lab.set_global_seed(123)
DEVICE = lab.choose_device(prefer_cuda=True)
if DEVICE.type != "cuda":
    raise RuntimeError("CUDA is required for this notebook.")
if "envs\\chess_engine" not in sys.executable.lower():
    raise RuntimeError(f"Wrong interpreter: {sys.executable}")

paths = lab.build_default_paths(run_dir=RUN_DIR, data_root=DATA_ROOT, experiment_dir=EXPERIMENT_DIR)
lab.export_paths_json(paths, paths["output_dir"] / "paths.json")

ORACLE_CFG = lab.OracleMineConfig()
PILOT_CFG = lab.PilotTrainConfig()
GATE_CFG = lab.MidbandGateConfig()
lab.validate_oracle_mine_config(ORACLE_CFG)
lab.validate_pilot_train_config(PILOT_CFG)
lab.validate_gate_config(GATE_CFG)
runtime_check = lab.validate_runtime_paths(paths, ORACLE_CFG)
if not runtime_check["ok"]:
    raise RuntimeError(runtime_check["issues"])

L4_CHECKPOINT = paths["objective_output_dir"] / "runs" / "L4_A1_plus_A2" / "checkpoints" / "L4_A1_plus_A2_best.pt"
REFERENCE = lab.build_reference_context(DATA_ROOT, paths, refresh=False)

lab.save_json(asdict(ORACLE_CFG), paths["output_dir"] / "oracle_cfg_initial.json")
lab.save_json(asdict(PILOT_CFG), paths["output_dir"] / "pilot_cfg_initial.json")
lab.save_json(asdict(GATE_CFG), paths["output_dir"] / "gate_cfg.json")
lab.save_json(runtime_check, paths["reports_dir"] / "runtime_check.json")

display(pd.DataFrame({"path_key": list(paths.keys()), "path_value": [str(v) for v in paths.values()]}))
display(pd.DataFrame([runtime_check]))

,path_key,path_value
0,project_root,C:\Users\USER\Desktop\chess_engine
1,run_dir,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...
2,data_root,C:\Users\USER\Desktop\chess_engine\data\process
3,experiment_dir,C:\Users\USER\Desktop\chess_engine\experiments...
4,objective_output_dir,C:\Users\USER\Desktop\chess_engine\experiments...
5,failure_b_output_dir,C:\Users\USER\Desktop\chess_engine\experiments...
6,failure_b_reports_dir,C:\Users\USER\Desktop\chess_engine\experiments...
7,output_dir,C:\Users\USER\Desktop\chess_engine\experiments...
8,reports_dir,C:\Users\USER\Desktop\chess_engine\experiments...
9,checkpoints_dir,C:\Users\USER\Desktop\chess_engine\experiments...


,ok,issues,baseline_checkpoint,a2_checkpoint,l0_checkpoint,l4_checkpoint,stockfish_path
0,True,[],C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...,C:\Users\USER\Desktop\chess_engine\experiments...,C:\Users\USER\Desktop\chess_engine\experiments...,C:\Users\USER\Desktop\chess_engine\experiments...,D:\stockfish-windows-x86-64-avx2\stockfish\sto...


## Context From Existing Suites

Cell này đọc lại đúng các bảng đã chốt trước khi chạy OC2:

- `baseline / A2 / L0 / L4`
- Failure B score hiện tại
- midband metrics hiện tại

In [2]:
failure_b_primary = pd.read_csv(paths["failure_b_reports_dir"] / "combined_failure_b_primary_metrics.csv")
objective_primary = pd.read_csv(paths["objective_output_dir"] / "reports" / "full_primary_metrics.csv")

display(
    failure_b_primary[
        failure_b_primary["label"].isin(["baseline", "A2_band_balanced", "L0_control_hybrid", "L4_A1_plus_A2"])
    ].sort_values("failure_b_score")
)
display(
    objective_primary[
        objective_primary["label"].isin(["baseline", "A2_band_balanced", "L0_control_hybrid", "L4_A1_plus_A2"])
    ][["label", "oracle_midband_mae_sum_stable", "oracle_stable_0.7_slope", "oracle_center_amp_ratio", "selection_score_v2"]]
    .sort_values("oracle_midband_mae_sum_stable")
)

,label,target_scale,metric_scale,test_mse_0.1eq,test_mse_0.2eq,test_mse_0.5eq,test_mse_0.7eq,test_slope_0.1eq,test_slope_0.2eq,test_slope_0.7eq,...,oracle_band_sign_0.2_0.5_stable,selection_score_v2,pooled_center_mae,pooled_center_amp_ratio,pooled_center_false_0.1eq,pooled_center_false_0.2eq,pooled_center_wrong_sign_0.1eq,pooled_center_wrong_sign_0.2eq,pooled_center_spread_ratio,failure_b_score
0,A2_band_balanced,600.0,600.0,0.029684,0.028911,0.036876,0.050254,1.283758,0.898517,0.594372,...,1.000000,1.462451,0.092978,4.145144,0.363636,0.136364,0.045455,0.0,4.354346,0.617330
1,L0_control_hybrid,600.0,600.0,0.031418,0.030474,0.038108,0.051023,1.336287,0.927135,0.608344,...,1.000000,1.471977,0.092319,4.189385,0.363636,0.136364,0.045455,0.0,4.406538,0.618233
2,baseline,600.0,600.0,0.032140,0.031037,0.038544,0.051428,1.338430,0.923810,0.605962,...,0.933333,1.502943,0.094196,4.317992,0.409091,0.136364,0.045455,0.0,4.468379,0.650608
4,L4_A1_plus_A2,600.0,600.0,0.039988,0.038093,0.044461,0.055779,1.436736,0.987802,0.645502,...,1.000000,1.551388,0.108972,4.799236,0.500000,0.181818,0.090909,0.0,5.045348,0.732969


,label,oracle_midband_mae_sum_stable,oracle_stable_0.7_slope,oracle_center_amp_ratio,selection_score_v2
9,L4_A1_plus_A2,0.570976,0.617916,6.379222,1.551388
5,L0_control_hybrid,0.590317,0.582414,5.683351,1.471977
2,A2_band_balanced,0.591547,0.569950,5.551442,1.462451
0,baseline,0.598789,0.575122,5.851215,1.502943


## Build L4 Train Prediction Cache

Cache này phục vụ mining 2D theo `raw |y| x |pred_L4|`.

In [3]:
pred_cache_manifest = lab.fb_lab.precompute_train_prediction_cache(
    checkpoint_path=L4_CHECKPOINT,
    data_root=DATA_ROOT,
    split="train",
    num_shards=ORACLE_CFG.train_num_shards,
    paths=paths,
    device=DEVICE,
    batch_size=ORACLE_CFG.prediction_cache_batch_size,
    refresh=False,
)

display(pd.DataFrame(pred_cache_manifest["shards"]).head())

[predict] offset=0 / 50000 elapsed=1.8s
[predict] offset=0 / 50000 elapsed=1.1s
[predict] offset=0 / 50000 elapsed=1.1s
[predict] offset=0 / 50000 elapsed=1.1s
[predict] offset=0 / 50000 elapsed=1.1s
[predict] offset=0 / 50000 elapsed=1.1s
[predict] offset=0 / 50000 elapsed=1.1s
[predict] offset=0 / 50000 elapsed=1.1s


,shard_id,pred_file,n
0,0,pred_00000.npy,50000
1,11,pred_00011.npy,50000
2,22,pred_00022.npy,50000
3,33,pred_00033.npy,50000
4,45,pred_00045.npy,50000


## Mine 2D Candidate Bundle

Candidate pool được chia đều theo hai trục:

- `raw |y|` band
- `|pred_L4|` band

Mục tiêu là tránh bias kiểu `OC1` chỉ tập trung vào các false-decisive hard case.

In [4]:
candidate_bundle = lab.build_2d_candidate_bundle(
    checkpoint_path=L4_CHECKPOINT,
    pred_cache_manifest=pred_cache_manifest,
    data_root=DATA_ROOT,
    cfg=ORACLE_CFG,
    paths=paths,
    refresh=False,
)

display(pd.DataFrame([candidate_bundle["manifest"]]))
display(candidate_bundle["quota_summary"])
display(candidate_bundle["rows"].head(12))

,schema_version,checkpoint_path,train_num_shards,raw_abs_y_edges,pred_abs_pred_edges,sample_per_cell,num_candidates
0,1,C:\Users\USER\Desktop\chess_engine\experiments...,8,"[0.0, 0.02, 0.05, 0.1]","[0.0, 0.1, 0.3, 0.6, 1.01]",16,192


,raw_band_idx,raw_band_label,pred_band_idx,pred_band_label,count,quota
0,0,"[0.000,0.020]",0,"[0.000,0.100]",41372,16
1,0,"[0.000,0.020]",1,"[0.100,0.300]",29413,16
2,0,"[0.000,0.020]",2,"[0.300,0.600]",8954,16
3,0,"[0.000,0.020]",3,"[0.600,1.010]",1155,16
4,1,"[0.020,0.050]",0,"[0.000,0.100]",25790,16
5,1,"[0.020,0.050]",1,"[0.100,0.300]",15022,16
6,1,"[0.020,0.050]",2,"[0.300,0.600]",3481,16
7,1,"[0.020,0.050]",3,"[0.600,1.010]",330,16
8,2,"[0.050,0.100]",0,"[0.000,0.100]",60101,16
9,2,"[0.050,0.100]",1,"[0.100,0.300]",88847,16


,candidate_id,shard_id,local_index,raw_band_idx,raw_band_label,pred_band_idx,pred_band_label,raw_target_y,raw_abs_y,init_pred,init_abs_pred,init_abs_err
0,72,33,28669,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.000000,0.000000,0.098389,0.098389,0.098389
1,170,79,34724,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.000000,0.000000,-0.091492,0.091492,0.091492
2,25,11,41845,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.018331,0.018331,-0.060028,0.060028,0.041697
3,97,45,24887,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.000000,0.000000,-0.037842,0.037842,0.037842
4,48,22,20533,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.000000,0.000000,-0.037201,0.037201,0.037201
5,49,22,39841,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.000000,0.000000,0.035950,0.035950,0.035950
6,121,56,40577,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.003333,0.003333,0.033936,0.033936,0.037269
7,169,79,26167,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.005000,0.005000,0.033264,0.033264,0.038264
8,96,45,19912,0,"[0.000,0.020]",0,"[0.000,0.100]",0.011666,0.011666,-0.031921,0.031921,0.043588
9,24,11,40039,0,"[0.000,0.020]",0,"[0.000,0.100]",0.008333,0.008333,0.027969,0.027969,0.019636


## Oracle Audit

Với mỗi candidate:

- decode về board
- chạy Stockfish multi-budget fixed-node
- lọc stable subset
- gán trusted center vs ambiguous roles theo oracle

In [5]:
oracle_audit = lab.run_oracle_candidate_audit(
    candidate_bundle=candidate_bundle,
    cfg=ORACLE_CFG,
    paths=paths,
    refresh=False,
)

display(pd.DataFrame([oracle_audit["report"]]))
display(oracle_audit["summary"])
display(
    oracle_audit["rows"][
        [
            "candidate_id",
            "raw_band_label",
            "pred_band_label",
            "raw_target_y",
            "init_pred",
            "oracle_final_y",
            "oracle_target_range",
            "oracle_bestmove_changes",
            "oracle_sign_flips",
            "is_stable",
            "is_center_clean",
            "aux_keep",
        ]
    ].head(20)
)

[oc2-oracle-audit] processed=1/192
[oc2-oracle-audit] processed=16/192
[oc2-oracle-audit] processed=32/192
[oc2-oracle-audit] processed=48/192
[oc2-oracle-audit] processed=64/192
[oc2-oracle-audit] processed=80/192
[oc2-oracle-audit] processed=96/192
[oc2-oracle-audit] processed=112/192
[oc2-oracle-audit] processed=128/192
[oc2-oracle-audit] processed=144/192
[oc2-oracle-audit] processed=160/192
[oc2-oracle-audit] processed=176/192
[oc2-oracle-audit] processed=192/192


,num_candidates,stable_count,center_clean_count,aux_keep_count,stable_rate,center_clean_rate,aux_keep_rate,trusted_center_thr,aux_oracle_abs_max
0,192,93,39,78,0.484375,0.203125,0.40625,0.05,0.25


,raw_band_idx,raw_band_label,pred_band_idx,pred_band_label,n,stable_count,center_clean_count,aux_keep_count,mean_init_abs_pred,mean_oracle_abs_y,mean_oracle_target_range
0,0,"[0.000,0.020]",0,"[0.000,0.100]",16,6,6,6,0.036751,0.014370,0.039996
1,0,"[0.000,0.020]",1,"[0.100,0.300]",16,7,5,7,0.156971,0.029342,0.044491
2,0,"[0.000,0.020]",2,"[0.300,0.600]",16,8,6,8,0.398285,0.039006,0.054703
3,0,"[0.000,0.020]",3,"[0.600,1.010]",16,9,8,9,0.727997,0.032013,0.041464
4,1,"[0.020,0.050]",0,"[0.000,0.100]",16,5,2,5,0.056142,0.056054,0.049700
5,1,"[0.020,0.050]",1,"[0.100,0.300]",16,8,3,8,0.188679,0.076268,0.052597
6,1,"[0.020,0.050]",2,"[0.300,0.600]",16,11,4,11,0.408157,0.062782,0.048103
7,1,"[0.020,0.050]",3,"[0.600,1.010]",16,8,3,8,0.709290,0.083072,0.088524
8,2,"[0.050,0.100]",0,"[0.000,0.100]",16,8,1,8,0.058662,0.179616,0.053719
9,2,"[0.050,0.100]",1,"[0.100,0.300]",16,9,0,6,0.175003,0.260177,0.079076


,candidate_id,raw_band_label,pred_band_label,raw_target_y,init_pred,oracle_final_y,oracle_target_range,oracle_bestmove_changes,oracle_sign_flips,is_stable,is_center_clean,aux_keep
0,72,"[0.000,0.020]","[0.000,0.100]",-0.000000,0.098389,0.000000,0.023330,2,2,False,False,False
1,170,"[0.000,0.020]","[0.000,0.100]",-0.000000,-0.091492,-0.018331,0.031664,0,1,False,False,False
2,25,"[0.000,0.020]","[0.000,0.100]",-0.018331,-0.060028,-0.041643,0.033217,1,0,True,True,True
3,97,"[0.000,0.020]","[0.000,0.100]",-0.000000,-0.037842,0.000000,0.000000,0,0,True,True,True
4,48,"[0.000,0.020]","[0.000,0.100]",-0.000000,-0.037201,0.000000,0.008333,0,1,False,False,False
5,49,"[0.000,0.020]","[0.000,0.100]",-0.000000,0.035950,-0.008333,0.028331,0,2,False,False,False
6,121,"[0.000,0.020]","[0.000,0.100]",-0.003333,0.033936,-0.016665,0.011665,2,0,False,False,False
7,169,"[0.000,0.020]","[0.000,0.100]",-0.005000,0.033264,0.001667,0.028331,1,2,False,False,False
8,96,"[0.000,0.020]","[0.000,0.100]",0.011666,-0.031921,0.029991,0.034933,0,0,True,True,True
9,24,"[0.000,0.020]","[0.000,0.100]",0.008333,0.027969,0.048296,0.011632,0,0,True,True,True


## Build Role Bundle

Role bundle tách 3 nhóm:

- `center_anchor`: oracle sạch gần 0 và `|pred_L4|` thấp
- `center_hard`: oracle sạch gần 0 nhưng `|pred_L4|` cao
- `center_ambiguous`: raw-center nhưng oracle không thật sự gần 0

In [6]:
role_bundle = lab.build_role_bundle(
    candidate_bundle=candidate_bundle,
    oracle_audit=oracle_audit,
    cfg=ORACLE_CFG,
    paths=paths,
    refresh=False,
)

display(pd.DataFrame([role_bundle["manifest"]]))
display(role_bundle["summary"])
display(role_bundle["rows"].head(12))

,schema_version,candidate_manifest,oracle_report,trusted_center_thr,aux_oracle_abs_max,center_anchor_pred_abs_max,num_rows,center_anchor_count,center_hard_count,center_ambiguous_count
0,1,"{'schema_version': 1, 'checkpoint_path': 'C:\U...","{'num_candidates': 192, 'stable_count': 93, 'c...",0.05,0.25,0.2,78,14,25,39


,role_code,role_name,raw_band_label,pred_band_label,n,mean_init_abs_pred,mean_oracle_abs_y
0,0,center_anchor,"[0.000,0.020]","[0.000,0.100]",6,0.033207,0.022488
1,0,center_anchor,"[0.000,0.020]","[0.100,0.300]",4,0.140335,0.009999
2,0,center_anchor,"[0.020,0.050]","[0.000,0.100]",2,0.021025,0.046633
3,0,center_anchor,"[0.020,0.050]","[0.100,0.300]",1,0.145752,0.044970
4,0,center_anchor,"[0.050,0.100]","[0.000,0.100]",1,0.061432,0.044970
5,1,center_hard,"[0.000,0.020]","[0.100,0.300]",1,0.292969,0.043306
6,1,center_hard,"[0.000,0.020]","[0.300,0.600]",6,0.396525,0.020829
7,1,center_hard,"[0.000,0.020]","[0.600,1.010]",8,0.717468,0.026030
8,1,center_hard,"[0.020,0.050]","[0.100,0.300]",2,0.261597,0.034982
9,1,center_hard,"[0.020,0.050]","[0.300,0.600]",4,0.365479,0.030820


,candidate_id,shard_id,local_index,raw_band_idx,raw_band_label,pred_band_idx,pred_band_label,raw_target_y,raw_abs_y,init_pred,...,oracle_bestmove_n4000,oracle_y_n16000,oracle_cp_n16000,oracle_bestmove_n16000,oracle_y_n64000,oracle_cp_n64000,oracle_bestmove_n64000,role_code,role_name,oracle_role_target_y
0,25,11,41845,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.018331,0.018331,-0.060028,...,f8b4,-0.043306,-26.0,f8e7,-0.041643,-25.0,f8e7,0,center_anchor,-0.041643
1,97,45,24887,0,"[0.000,0.020]",0,"[0.000,0.100]",-0.000000,0.000000,-0.037842,...,a6h6,0.000000,0.0,a6h6,0.000000,0.0,a6h6,0,center_anchor,0.000000
2,96,45,19912,0,"[0.000,0.020]",0,"[0.000,0.100]",0.011666,0.011666,-0.031921,...,c2c4,0.024995,15.0,c2c4,0.029991,18.0,c2c4,0,center_anchor,0.029991
3,24,11,40039,0,"[0.000,0.020]",0,"[0.000,0.100]",0.008333,0.008333,0.027969,...,h6h5,0.059928,36.0,h6h5,0.048296,29.0,h6h5,0,center_anchor,0.048296
4,1,0,33478,0,"[0.000,0.020]",0,"[0.000,0.100]",0.000000,0.000000,-0.027115,...,f1e1,0.013333,8.0,f1e1,0.001667,1.0,f1e1,0,center_anchor,0.001667
5,146,67,12029,0,"[0.000,0.020]",0,"[0.000,0.100]",0.018331,0.018331,-0.014366,...,f1b5,0.053283,32.0,f1b5,0.013333,8.0,f1b5,0,center_anchor,0.013333
6,2,0,2434,0,"[0.000,0.020]",1,"[0.100,0.300]",0.000000,0.000000,-0.292969,...,e5g3,-0.044970,-27.0,e5g3,-0.043306,-26.0,e5g3,1,center_hard,-0.043306
7,50,22,42951,0,"[0.000,0.020]",1,"[0.100,0.300]",-0.000000,0.000000,-0.173218,...,e7d7,-0.005000,-3.0,e7d7,-0.001667,-1.0,e7d6,0,center_anchor,-0.001667
8,26,11,25817,0,"[0.000,0.020]",1,"[0.100,0.300]",0.000000,0.000000,-0.139404,...,e5c6,0.001667,1.0,e5c6,0.016665,10.0,e5c6,0,center_anchor,0.016665
9,172,79,21979,0,"[0.000,0.020]",1,"[0.100,0.300]",-0.001667,0.001667,-0.134766,...,f3h5,-0.028326,-17.0,f3h5,-0.016665,-10.0,f3h5,0,center_anchor,-0.016665


## Autotune Joint Full-Model Batch Size

Khác với `OC1`, batch autotune ở đây dùng **joint step thật**:

- forward/backward main
- forward/backward aux
- một optimizer step duy nhất

In [7]:
AUTOTUNE = lab.autotune_joint_batch_size(
    init_ckpt_path=L4_CHECKPOINT,
    data_root=DATA_ROOT,
    role_bundle=role_bundle,
    pilot_cfg=PILOT_CFG,
    device=DEVICE,
    preferred_batch_size=PILOT_CFG.main_batch_size,
    min_batch_size=PILOT_CFG.min_batch_size,
    step=PILOT_CFG.batch_step,
    max_mem_ratio=PILOT_CFG.max_mem_ratio,
)

PILOT_CFG = replace(PILOT_CFG, main_batch_size=int(AUTOTUNE["selected_batch_size"]))
REFERENCE["eval_cfg"] = replace(
    REFERENCE["eval_cfg"],
    batch_size=int(min(REFERENCE["eval_cfg"].batch_size, max(PILOT_CFG.main_batch_size, 128))),
)
lab.validate_pilot_train_config(PILOT_CFG)
lab.save_json(AUTOTUNE, paths["reports_dir"] / "joint_batch_autotune.json")
lab.save_json(asdict(PILOT_CFG), paths["output_dir"] / "pilot_cfg_final.json")

preview_model, _ = lab.base_lab.load_model_from_checkpoint(L4_CHECKPOINT, device=DEVICE)
preview_scope = lab.configure_full_model_trainable(preview_model)
del preview_model
torch.cuda.empty_cache()

runtime_summary = pd.DataFrame(
    [
        {
            "python_executable": sys.executable,
            "device": str(DEVICE),
            "gpu_name": torch.cuda.get_device_name(0),
            "gpu_vram_gb": round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 3),
            "main_batch_size": int(PILOT_CFG.main_batch_size),
            "anchor_batch_size": int(PILOT_CFG.anchor_batch_size),
            "hard_batch_size": int(PILOT_CFG.hard_batch_size),
            "ambiguous_batch_size": int(PILOT_CFG.ambiguous_batch_size),
            "epochs": int(PILOT_CFG.epochs),
        }
    ]
)

display(runtime_summary)
display(pd.DataFrame(AUTOTUNE["attempts"]))
display(preview_scope)

,python_executable,device,gpu_name,gpu_vram_gb,main_batch_size,anchor_batch_size,hard_batch_size,ambiguous_batch_size,epochs
0,c:\Users\USER\anaconda3\envs\chess_engine\pyth...,cuda,NVIDIA GeForce RTX 2050,4.0,384,24,24,48,1


,batch_size,ok,peak_mem_gb,mem_ratio,epoch_hours_estimate
0,384,True,2.20545,0.55143,0.567259


,module,total_params,trainable_params,trainable
0,stem,41984,41984,True
1,blocks.0,386656,386656,True
2,blocks.1,386656,386656,True
3,blocks.2,386656,386656,True
4,blocks.3,386656,386656,True
5,blocks.4,386656,386656,True
6,blocks.5,386656,386656,True
7,blocks.6,386656,386656,True
8,blocks.7,386656,386656,True
9,blocks.8,386656,386656,True


## Fresh L4 Reference Under OC2 Eval Setup

Cell này chấm lại `L4` bằng đúng pipeline sẽ dùng để gate OC2.

In [8]:
l4_reference = lab.evaluate_l4_reference(
    checkpoint_path=L4_CHECKPOINT,
    data_root=DATA_ROOT,
    pooled_center_bundle=REFERENCE["pooled_center_bundle"],
    oracle_bundle=REFERENCE["oracle_bundle"],
    eval_cfg=REFERENCE["eval_cfg"],
    paths=paths,
    device=DEVICE,
    prefix="l4_reference",
)

display(pd.DataFrame([l4_reference["primary"]]))
display(pd.DataFrame([l4_reference["center_eval"]]))
display(pd.DataFrame([{"center_score": l4_reference["center_score"], "legacy_failure_b_score": l4_reference["legacy_failure_b_score"]}]))

[test_600] offset=0 / 200000 elapsed=0.5s
[test_600] offset=25600 / 200000 elapsed=13.8s
[test_600] offset=51200 / 200000 elapsed=27.0s
[test_600] offset=76800 / 200000 elapsed=40.2s
[test_600] offset=102400 / 200000 elapsed=53.5s
[test_600] offset=128000 / 200000 elapsed=66.8s
[test_600] offset=153600 / 200000 elapsed=80.0s
[test_600] offset=179200 / 200000 elapsed=93.3s
[oracle_subset_600] offset=0 / 240 elapsed=0.1s


,label,target_scale,metric_scale,test_mse_0.1eq,test_mse_0.2eq,test_mse_0.5eq,test_mse_0.7eq,test_slope_0.1eq,test_slope_0.2eq,test_slope_0.7eq,...,oracle_band_mae_0.05_0.2_stable,oracle_band_mae_0.2_0.5_stable,oracle_band_mae_0.5_0.7_stable,oracle_band_amp_0_0.05_stable,oracle_band_amp_0.2_0.5_stable,oracle_band_amp_0.5_0.7_stable,oracle_band_sign_0_0.05_stable,oracle_band_sign_0.05_0.2_stable,oracle_band_sign_0.2_0.5_stable,selection_score_v2
0,L4_A1_plus_A2,600.0,600.0,0.039988,0.038093,0.044461,0.055779,1.436736,0.987802,0.645502,...,0.129522,0.188913,0.25254,5.685617,0.632936,0.677562,0.25,0.583333,1.0,1.551388


,label,checkpoint,n,mae_vs_oracle,amp_ratio,false_decisive_0.1,false_decisive_0.2,wrong_sign_0.1,wrong_sign_0.2,spread_ratio,mean_abs_pred,mean_abs_oracle
0,L4_A1_plus_A2,C:\Users\USER\Desktop\chess_engine\experiments...,22,0.108972,4.799236,0.5,0.181818,0.090909,0.0,5.045348,0.109745,0.022867


,center_score,legacy_failure_b_score
0,0.525259,0.732969


## Run OC2 Pilot

Training logic:

- full-model trainable
- `main_loss = L4 + smooth raw-center downweight`
- `aux_loss = oracle anchor + oracle hard + oracle ambiguous + center margin`
- accumulate gradients from `main` và `aux`, rồi mới `optimizer.step()`
- epoch-end eval luôn ở `model.eval()`

In [9]:
pilot = lab.run_oc2_joint_oracle_full_model_pilot(
    init_ckpt_path=L4_CHECKPOINT,
    data_root=DATA_ROOT,
    pilot_cfg=PILOT_CFG,
    gate_cfg=GATE_CFG,
    role_bundle=role_bundle,
    oracle_bundle=REFERENCE["oracle_bundle"],
    pooled_center_bundle=REFERENCE["pooled_center_bundle"],
    l4_reference=l4_reference,
    paths=paths,
    device=DEVICE,
)

display(pilot["trainable_scope"])
display(pilot["history"])
display(pd.DataFrame([pilot["decision_summary"]]))

[oc2-pilot] finished shard 1/8
[oc2-pilot] step=200/1042 main_obj=0.081320 aux_obj=0.016975
[oc2-pilot] finished shard 2/8
[oc2-pilot] finished shard 3/8
[oc2-pilot] step=400/1042 main_obj=0.075332 aux_obj=0.009644
[oc2-pilot] finished shard 4/8
[oc2-pilot] step=600/1042 main_obj=0.077047 aux_obj=0.007085
[oc2-pilot] finished shard 5/8
[oc2-pilot] finished shard 6/8
[oc2-pilot] step=800/1042 main_obj=0.074564 aux_obj=0.005767
[oc2-pilot] finished shard 7/8
[oc2-pilot] step=1000/1042 main_obj=0.075313 aux_obj=0.004954
[oc2-pilot] finished shard 8/8
[test_600] offset=0 / 200000 elapsed=0.5s
[test_600] offset=25600 / 200000 elapsed=13.9s
[test_600] offset=51200 / 200000 elapsed=27.2s
[test_600] offset=76800 / 200000 elapsed=40.5s
[test_600] offset=102400 / 200000 elapsed=53.9s
[test_600] offset=128000 / 200000 elapsed=67.2s
[test_600] offset=153600 / 200000 elapsed=80.5s
[test_600] offset=179200 / 200000 elapsed=93.7s
[oracle_subset_600] offset=0 / 240 elapsed=0.1s
{
  "epoch": 0,
  "trai

,module,total_params,trainable_params,trainable
0,stem,41984,41984,True
1,blocks.0,386656,386656,True
2,blocks.1,386656,386656,True
3,blocks.2,386656,386656,True
4,blocks.3,386656,386656,True
5,blocks.4,386656,386656,True
6,blocks.5,386656,386656,True
7,blocks.6,386656,386656,True
8,blocks.7,386656,386656,True
9,blocks.8,386656,386656,True


,epoch,train_main_objective,train_main_term,train_mean_main_weight,train_downweighted_frac,train_aux_objective,train_aux_anchor_loss,train_aux_hard_loss,train_aux_ambiguous_loss,train_aux_margin,...,oracle_stable_0.7_slope,pooled_center_mae,pooled_center_amp_ratio,pooled_center_false_0.1eq,pooled_center_false_0.2eq,center_score,legacy_failure_b_score,midband_gate_pass,lr,epoch_time_sec
0,0,0.073737,0.067347,0.80813,0.429465,0.004809,0.000798,0.003719,0.01033,0.007647,...,0.644228,0.124083,5.430452,0.5,0.181818,0.603492,0.807004,True,8.000982e-07,981.113971


,best_any_center_score,best_gate_center_score,has_gate_checkpoint,l4_center_score,l4_legacy_failure_b_score,l4_midband_mae,l4_stable_slope
0,0.603492,0.603492,True,0.525259,0.732969,0.570976,0.617916


## Final Comparison Against Existing Checkpoints

So sánh:

- `baseline`
- `A2`
- `L0`
- `L4`
- `OC2_best_any_center`
- `OC2_best_gate` nếu có

In [10]:
compare = lab.evaluate_registry_with_oc2(
    registry=REFERENCE["registry"],
    oc2_best_any_checkpoint=pilot["best_any_checkpoint"],
    oc2_best_gate_checkpoint=pilot["best_gate_checkpoint"],
    data_root=DATA_ROOT,
    pooled_center_bundle=REFERENCE["pooled_center_bundle"],
    oracle_bundle=REFERENCE["oracle_bundle"],
    eval_cfg=REFERENCE["eval_cfg"],
    l4_reference=l4_reference,
    gate_cfg=GATE_CFG,
    paths=paths,
    device=DEVICE,
    prefix="combined_oc2_final_pilot",
)

display(compare["primary"])
display(compare["pooled_center"])

[test_600] offset=0 / 200000 elapsed=0.5s
[test_600] offset=25600 / 200000 elapsed=13.8s
[test_600] offset=51200 / 200000 elapsed=27.1s
[test_600] offset=76800 / 200000 elapsed=40.3s
[test_600] offset=102400 / 200000 elapsed=53.6s
[test_600] offset=128000 / 200000 elapsed=66.9s
[test_600] offset=153600 / 200000 elapsed=80.2s
[test_600] offset=179200 / 200000 elapsed=93.5s
[oracle_subset_600] offset=0 / 240 elapsed=0.1s
[test_600] offset=0 / 200000 elapsed=0.5s
[test_600] offset=25600 / 200000 elapsed=13.8s
[test_600] offset=51200 / 200000 elapsed=27.1s
[test_600] offset=76800 / 200000 elapsed=40.4s
[test_600] offset=102400 / 200000 elapsed=53.7s
[test_600] offset=128000 / 200000 elapsed=67.0s
[test_600] offset=153600 / 200000 elapsed=80.3s
[test_600] offset=179200 / 200000 elapsed=93.6s
[oracle_subset_600] offset=0 / 240 elapsed=0.1s
[test_600] offset=0 / 200000 elapsed=0.6s
[test_600] offset=25600 / 200000 elapsed=14.0s
[test_600] offset=51200 / 200000 elapsed=27.3s
[test_600] offset=

,label,target_scale,metric_scale,test_mse_0.1eq,test_mse_0.2eq,test_mse_0.5eq,test_mse_0.7eq,test_slope_0.1eq,test_slope_0.2eq,test_slope_0.7eq,...,oracle_band_sign_0.05_0.2_stable,oracle_band_sign_0.2_0.5_stable,selection_score_v2,pooled_center_mae,pooled_center_amp_ratio,pooled_center_false_0.1eq,pooled_center_false_0.2eq,center_score,legacy_failure_b_score,midband_gate_pass
0,A2_band_balanced,600.0,600.0,0.029684,0.028911,0.036876,0.050254,1.283758,0.898517,0.594372,...,0.625000,1.000000,1.462451,0.092978,4.145145,0.363636,0.136364,0.393856,0.617330,False
1,L0_control_hybrid,600.0,600.0,0.031418,0.030474,0.038108,0.051023,1.336287,0.927135,0.608344,...,0.625000,1.000000,1.471977,0.092319,4.189385,0.363636,0.136364,0.397621,0.618233,False
2,baseline,600.0,600.0,0.032140,0.031037,0.038544,0.051428,1.338430,0.923810,0.605962,...,0.625000,0.933333,1.502943,0.094196,4.317992,0.409091,0.136364,0.425995,0.650608,False
3,L4_A1_plus_A2,600.0,600.0,0.039988,0.038093,0.044461,0.055779,1.436736,0.987802,0.645502,...,0.583333,1.000000,1.551388,0.108972,4.799236,0.500000,0.181818,0.525259,0.732969,True
4,OC2_best_any_center,600.0,600.0,0.051825,0.048922,0.054245,0.064271,1.567104,1.064547,0.672451,...,0.625000,0.933333,1.651929,0.124083,5.430452,0.500000,0.181818,0.603492,0.807004,True
5,OC2_best_gate,600.0,600.0,0.051825,0.048922,0.054245,0.064271,1.567104,1.064547,0.672451,...,0.625000,0.933333,1.651929,0.124083,5.430452,0.500000,0.181818,0.603492,0.807004,True


,label,checkpoint,n,mae_vs_oracle,amp_ratio,false_decisive_0.1,false_decisive_0.2,wrong_sign_0.1,wrong_sign_0.2,spread_ratio,mean_abs_pred,mean_abs_oracle
0,baseline,C:\Users\USER\Downloads\dgrn_5m_v3_stage2_poli...,22,0.094196,4.317992,0.409091,0.136364,0.045455,0.0,4.468379,0.098740,0.022867
1,A2_band_balanced,C:\Users\USER\Desktop\chess_engine\experiments...,22,0.092978,4.145145,0.363636,0.136364,0.045455,0.0,4.354347,0.094788,0.022867
2,L0_control_hybrid,C:\Users\USER\Desktop\chess_engine\experiments...,22,0.092319,4.189385,0.363636,0.136364,0.045455,0.0,4.406538,0.095800,0.022867
3,L4_A1_plus_A2,C:\Users\USER\Desktop\chess_engine\experiments...,22,0.108972,4.799236,0.500000,0.181818,0.090909,0.0,5.045348,0.109745,0.022867
4,OC2_best_any_center,C:\Users\USER\Desktop\chess_engine\experiments...,22,0.124083,5.430452,0.500000,0.181818,0.136364,0.0,5.528787,0.124179,0.022867
5,OC2_best_gate,C:\Users\USER\Desktop\chess_engine\experiments...,22,0.124083,5.430452,0.500000,0.181818,0.136364,0.0,5.528787,0.124179,0.022867


## Decision Rule

Thí nghiệm được xem là thành công nếu:

1. `OC2_best_gate` tồn tại
2. `center_score` giảm rõ rệt so với `L4`
3. `oracle_midband_mae_sum_stable` và `oracle_stable_0.7_slope` vẫn pass hard gate

Nếu `OC2_best_any_center` tốt hơn center nhưng không có `best_gate`, điều đó nghĩa là tín hiệu sửa B có tồn tại nhưng vẫn phá A quá mức.

In [ ]:
compare_primary = compare["primary"].copy()
decision_frame = compare_primary[
    [
        "label",
        "center_score",
        "legacy_failure_b_score",
        "oracle_midband_mae_sum_stable",
        "oracle_stable_0.7_slope",
        "midband_gate_pass",
    ]
].sort_values(["center_score", "oracle_midband_mae_sum_stable"], ascending=[True, True])

display(decision_frame)

,label,center_score,legacy_failure_b_score,oracle_midband_mae_sum_stable,oracle_stable_0.7_slope,midband_gate_pass
0,A2_band_balanced,0.393856,0.617330,0.591547,0.569950,False
1,L0_control_hybrid,0.397621,0.618233,0.590317,0.582414,False
2,baseline,0.425995,0.650608,0.598789,0.575122,False
3,L4_A1_plus_A2,0.525259,0.732969,0.570976,0.617916,True
4,OC2_best_any_center,0.603492,0.807004,0.574525,0.644228,True
5,OC2_best_gate,0.603492,0.807004,0.574525,0.644228,True


: 